# Analytics as Polars Expression Plugins

This notebook demonstrates the **expression plugin** API for finstack analytics.
Instead of constructing a `Performance` object and calling methods, you can
compute risk metrics, transforms, and benchmark-relative statistics directly
inside Polars `.select()` and `.with_columns()` calls.

**Learning objectives:**
- Compute 25+ risk metrics in a single `df.select(...)` call
- Transform prices to returns, cumulative returns, and drawdowns as Polars expressions
- Calculate benchmark-relative analytics (beta, tracking error, information ratio)
- Build rolling analytics with rolling Sharpe and volatility
- Understand when to use expression plugins vs the Performance class

In [1]:
import math
from datetime import date, timedelta

import polars as pl

from finstack.core.analytics import Performance, expr

## Sample Data

We generate deterministic price paths for two stocks and a benchmark:

In [2]:
n = 500
base_date = date(2023, 1, 2)
dates = [base_date + timedelta(days=i) for i in range(n)]

price_aapl = [150.0]
price_msft = [250.0]
price_spy = [400.0]
for i in range(1, n):
    price_aapl.append(price_aapl[-1] * (1.0 + 0.001 * math.sin(i * 0.3) + 0.0004))
    price_msft.append(price_msft[-1] * (1.0 + 0.0008 * math.cos(i * 0.2) + 0.0003))
    price_spy.append(price_spy[-1] * (1.0 + 0.0002))

prices = pl.DataFrame(
    {"date": dates, "AAPL": price_aapl, "MSFT": price_msft, "SPY": price_spy}
).with_columns(pl.col("date").cast(pl.Date))

prices.head()

date,AAPL,MSFT,SPY
date,f64,f64,f64
2023-01-02,150.0,250.0,400.0
2023-01-03,150.104328,250.271013,400.08
2023-01-04,150.249125,250.530507,400.160016
2023-01-05,150.426919,250.771083,400.240048
2023-01-06,150.627293,250.986086,400.320096


## Tier 1: Scalar Risk Metrics

Aggregation expressions collapse a column into a single scalar value.
You can compute dozens of metrics in one `.select()` — Polars runs them
in parallel on native Rust code with zero Python overhead.

In [3]:
# First convert prices to returns, dropping the leading zero
returns = prices.select(
    expr.simple_returns("AAPL").alias("AAPL"),
    expr.simple_returns("MSFT").alias("MSFT"),
    expr.simple_returns("SPY").alias("SPY"),
).slice(1)

# Compute a full risk dashboard in a single call
risk_dashboard = returns.select(
    expr.sharpe("AAPL", freq="daily").alias("AAPL Sharpe"),
    expr.sharpe("MSFT", freq="daily").alias("MSFT Sharpe"),
    expr.sortino("AAPL", freq="daily").alias("AAPL Sortino"),
    expr.sortino("MSFT", freq="daily").alias("MSFT Sortino"),
    expr.volatility("AAPL", freq="daily").alias("AAPL Vol"),
    expr.volatility("MSFT", freq="daily").alias("MSFT Vol"),
    expr.max_drawdown("AAPL").alias("AAPL Max DD"),
    expr.max_drawdown("MSFT").alias("MSFT Max DD"),
)
risk_dashboard

AAPL Sharpe,MSFT Sharpe,AAPL Sortino,MSFT Sortino,AAPL Vol,MSFT Vol,AAPL Max DD,MSFT Max DD
f64,f64,f64,f64,f64,f64,f64,f64
9.012836,8.279498,24.375736,21.125428,0.01126,0.008962,-0.003033,-0.003862


In [4]:
# Additional metrics: VaR, ES, higher moments, and ratio-based measures
advanced_metrics = returns.select(
    expr.value_at_risk("AAPL", confidence=0.95).alias("VaR 95%"),
    expr.expected_shortfall("AAPL", confidence=0.95).alias("ES 95%"),
    expr.skewness("AAPL").alias("Skewness"),
    expr.kurtosis("AAPL").alias("Kurtosis"),
    expr.omega_ratio("AAPL", threshold=0.0).alias("Omega"),
    expr.gain_to_pain("AAPL").alias("Gain-to-Pain"),
    expr.ulcer_index("AAPL").alias("Ulcer Index"),
    expr.recovery_factor("AAPL").alias("Recovery Factor"),
    expr.risk_of_ruin("AAPL", freq="daily").alias("Risk of Ruin"),
    expr.geometric_mean("AAPL").alias("Geo Mean"),
)
advanced_metrics

VaR 95%,ES 95%,Skewness,Kurtosis,Omega,Gain-to-Pain,Ulcer Index,Recovery Factor,Risk of Ruin,Geo Mean
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.000588,-0.000595,-0.008791,-1.507147,3.785385,2.785385,0.001509,73.315141,0.0,0.000402


## Tier 2: Series Transforms

These expressions produce a new column of the same length as the input.
Use them with `.with_columns()` to enrich your DataFrame.

In [5]:
enriched = prices.with_columns(
    expr.simple_returns("AAPL").alias("returns"),
    expr.cumulative_returns(expr.simple_returns("AAPL")).alias("cumulative"),
    expr.drawdown_series(expr.simple_returns("AAPL")).alias("drawdown"),
)

enriched.select("date", "AAPL", "returns", "cumulative", "drawdown").head(10)

date,AAPL,returns,cumulative,drawdown
date,f64,f64,f64,f64
2023-01-02,150.0,0.0,0.0,0.0
2023-01-03,150.104328,0.000696,0.000696,0.0
2023-01-04,150.249125,0.000965,0.001661,0.0
2023-01-05,150.426919,0.001183,0.002846,0.0
2023-01-06,150.627293,0.001332,0.004182,0.0
2023-01-07,150.837794,0.001397,0.005585,0.0
2023-01-08,151.045022,0.001374,0.006967,0.0
2023-01-09,151.235824,0.001263,0.008239,0.0
2023-01-10,151.398472,0.001075,0.009323,0.0


## Tier 3: Benchmark-Relative Metrics

Two-input expressions take a portfolio column and a benchmark column.
They measure how a portfolio behaves *relative to* the benchmark.

In [6]:
benchmark_analysis = returns.select(
    expr.tracking_error("AAPL", "SPY", freq="daily").alias("AAPL TE"),
    expr.tracking_error("MSFT", "SPY", freq="daily").alias("MSFT TE"),
    expr.beta("AAPL", "SPY").alias("AAPL Beta"),
    expr.beta("MSFT", "SPY").alias("MSFT Beta"),
    expr.information_ratio("AAPL", "SPY", freq="daily").alias("AAPL IR"),
    expr.information_ratio("MSFT", "SPY", freq="daily").alias("MSFT IR"),
    expr.r_squared("AAPL", "SPY").alias("AAPL R²"),
    expr.batting_average("AAPL", "SPY").alias("AAPL BA"),
)
benchmark_analysis

AAPL TE,MSFT TE,AAPL Beta,MSFT Beta,AAPL IR,MSFT IR,AAPL R²,AAPL BA
f64,f64,f64,f64,f64,f64,f64,f64
0.01126,0.008962,0.0,0.0,4.536746,2.656017,0.0,0.563126


## Tier 4: Rolling Analytics

Rolling expressions produce a series the same length as the input, with
NaN values for the initial warm-up period (before the first full window).

In [7]:
rolling_df = returns.with_columns(
    expr.rolling_sharpe("AAPL", window=60, freq="daily").alias("rolling_sharpe"),
    expr.rolling_volatility("AAPL", window=60, freq="daily").alias("rolling_vol"),
)

# Show data after warm-up period
rolling_df.select("AAPL", "rolling_sharpe", "rolling_vol").tail(10)

AAPL,rolling_sharpe,rolling_vol
f64,f64,f64
0.001009,8.644229,0.011554
0.000747,8.958887,0.011533
0.000455,9.318602,0.011435
0.000157,9.678011,0.01129
-0.000119,9.975018,0.01115
-0.000348,10.145102,0.011065
-0.000511,10.145619,0.011064
-0.000592,9.976432,0.01115
-0.000585,9.679986,0.011289


## Expression Plugins vs Performance Class

Both APIs call the same Rust core functions — results are identical.

| Use case | API | Why |
|---|---|---|
| Ad-hoc metric in a Polars pipeline | `expr.sharpe("col")` | Composable, no object construction |
| Full portfolio workflow | `Performance(df)` | Stateful, all metrics pre-aligned |
| Rolling or drawdown enrichment | `expr.rolling_sharpe("col")` | Column-level, auto-parallel |
| Multi-ticker comparison report | `Performance(df)` | Handles benchmark alignment |

In [8]:
perf = Performance(prices, benchmark_ticker="SPY", freq="daily")
perf_sharpe = perf.sharpe()

expr_sharpe = returns.select(
    expr.sharpe("AAPL", freq="daily").alias("AAPL"),
    expr.sharpe("MSFT", freq="daily").alias("MSFT"),
)

print("Performance class:")
print(perf_sharpe)
print()
print("Expression plugin:")
print(expr_sharpe)

# Verify exact match
perf_val = perf_sharpe.filter(pl.col("ticker") == "AAPL")["sharpe"].item()
expr_val = expr_sharpe["AAPL"].item()
assert abs(perf_val - expr_val) < 1e-10
print(f"\nSharpe ratios match exactly: {expr_val:.6f}")

Performance class:
shape: (3, 2)
┌────────┬──────────┐
│ ticker ┆ sharpe   │
│ ---    ┆ ---      │
│ str    ┆ f64      │
╞════════╪══════════╡
│ AAPL   ┆ 9.012836 │
│ MSFT   ┆ 8.279498 │
│ SPY    ┆ 0.0      │
└────────┴──────────┘

Expression plugin:
shape: (1, 2)
┌──────────┬──────────┐
│ AAPL     ┆ MSFT     │
│ ---      ┆ ---      │
│ f64      ┆ f64      │
╞══════════╪══════════╡
│ 9.012836 ┆ 8.279498 │
└──────────┴──────────┘

Sharpe ratios match exactly: 9.012836


## Advanced: Composing with Polars

Because expression plugins return standard `pl.Expr`, they compose with
all Polars operations: `.filter()`, `.group_by()`, `.over()`, etc.

In [9]:
# Filter to high-drawdown periods, then compute metrics
deep_dd = enriched.filter(
    pl.col("drawdown") < -0.005
)
if deep_dd.height > 0:
    stress_metrics = deep_dd.select(
        pl.col("drawdown").min().alias("max_drawdown_in_period"),
        pl.col("drawdown").mean().alias("avg_drawdown_in_period"),
    )
    print("Stress period metrics:")
    print(stress_metrics)
else:
    print("No deep drawdown periods found in this synthetic data.")

No deep drawdown periods found in this synthetic data.
